<a href="https://colab.research.google.com/github/rahulpirwani7/Student-Pass-Fail-Prediction/blob/main/Passandfailpredication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("kundanbedmutha/exam-score-prediction-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'exam-score-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/exam-score-prediction-dataset


In [ ]:
files = os.listdir(path)

print("\nFiles in the dataset:")
for file in files:
    print(file)

csv_file = None
for file in files:
    if file.endswith(".csv"):
        csv_file = file
        break

if csv_file is None:
    print("No CSV file found in the dataset.")
else:
    file_path = os.path.join(path, csv_file)
    df = pd.read_csv(file_path)


Files in the dataset:
Exam_Score_Prediction.csv


In [ ]:
df.columns=df.columns.tolist()
df.drop('student_id',axis=1,inplace=True)


In [ ]:
df.describe()

,age,study_hours,class_attendance,sleep_hours,exam_score
count,20000.000000,20000.000000,20000.000000,20000.00000,20000.000000
mean,20.473300,4.007604,70.017365,7.00856,62.513225
std,2.284458,2.308313,17.282262,1.73209,18.908491
min,17.000000,0.080000,40.600000,4.10000,19.599000
25%,18.000000,2.000000,55.100000,5.50000,48.800000
50%,20.000000,4.040000,69.900000,7.00000,62.600000
75%,22.000000,6.000000,85.000000,8.50000,76.300000
max,24.000000,7.910000,99.400000,9.90000,100.000000


In [ ]:
X=['gender','study_method','course']
df = pd.get_dummies(
    df,
    columns=X,
    prefix="",
    prefix_sep="",
    dtype=int
)

In [ ]:
df["Passed"] = df["exam_score"].apply(lambda x: "Passed" if x >= 65 else "Failed")
df.drop("exam_score", axis=1, inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
X=['internet_access','Passed']
for i in X:
    df[i]=le.fit_transform(df[i])

df["sleep_quality"] = df["sleep_quality"].map({
    "poor": 0,
    "average": 1,
    "good": 2
})

df['facility_rating']=df['facility_rating'].map({
    'low':0,
    'medium':1,
    'high':2
})

df['exam_difficulty']=df['exam_difficulty'].map({
    'easy':0,
    'moderate':1,
    'hard':2
})


In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop('Passed', axis=1)
y = df['Passed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Original shape: (14000, 23)
New shape: (14000, 23)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [232]:

model = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
     eta0=1e-2,
    max_iter=1,
    tol=None,
    random_state=2
)

model.fit(X_train, y_train)

SGDClassifier(eta0=0.01, learning_rate='constant', loss='log_loss', max_iter=1,
              random_state=2, tol=None)

In [233]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_prob = model.predict_proba(X_train)[:, 1]

threshold=0.5

print(y_prob)
y_pred = (y_prob >= threshold).astype(int)

print(40*" "+f"Threshold = {threshold}")
print(40*" "+"Accuracy :", accuracy_score(y_train, y_pred))
print(40*" "+"Precision:", precision_score(y_train, y_pred))
print(40*" "+"Recall   :", recall_score(y_train, y_pred))
print(40*" "+"F1-score :", f1_score(y_train, y_pred))
print("-" * 100)

[0.04733783 0.10055605 0.00725829 ... 0.1205758  0.01127133 0.08301292]
                                        Threshold = 0.5
                                        Accuracy : 0.8345714285714285
                                        Precision: 0.8174344656429346
                                        Recall   : 0.8222811671087533
                                        F1-score : 0.8198506533914126
----------------------------------------------------------------------------------------------------


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_prob = model.predict_proba(X_test)[:, 1]

threshold=0.5
y_pred = (y_prob >= threshold).astype(int)

print(40*" "+f"Threshold = {threshold}")
print(40*" "+"Accuracy :", accuracy_score(y_test, y_pred))
print(40*" "+"Precision:", precision_score(y_test, y_pred))
print(40*" "+"Recall   :", recall_score(y_test, y_pred))
print(40*" "+"F1-score :", f1_score(y_test, y_pred))
print("-" * 100)

                                        Threshold = 0.5
                                        Accuracy : 0.842
                                        Precision: 0.8275101140125046
                                        Recall   : 0.8244778307072188
                                        F1-score : 0.8259911894273128
----------------------------------------------------------------------------------------------------


In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

[[2802  469]
 [ 479 2250]]


In [ ]:
df["Passed"].value_counts()

,count
Passed,
0,10862
1,9138


In [ ]:
print(y_train.value_counts())

Passed
0    7591
1    6409
Name: count, dtype: int64


In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(14000, 23)
(6000, 23)
(14000,)
(6000,)


AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [196]:
print(df.corr(numeric_only=True)["Passed"].sort_values(ascending=False))

Passed              1.000000
study_hours         0.621829
class_attendance    0.240073
sleep_quality       0.149033
coaching            0.121283
facility_rating     0.120706
sleep_hours         0.095664
mixed               0.036206
bba                 0.011000
b.sc                0.007161
age                 0.005184
bca                 0.004012
exam_difficulty     0.003840
male                0.002351
b.tech              0.002197
female             -0.000416
other              -0.001935
internet_access    -0.003316
ba                 -0.006612
diploma            -0.006973
b.com              -0.010763
group study        -0.033866
online videos      -0.047405
self-study         -0.075653
Name: Passed, dtype: float64
